# <img align="left" src="./图片/film_strip_vertical.png"     style=" width:40px;  " > 实践实验：基于内容的过滤深度学习

在本练习中，你将用神经网络实现基于内容的过滤，为电影构建推荐系统。

# 目录 <img align="left" src="./图片/film_reel.png"     style=" width:40px;  " >
- [ 1 - 程序包](#1)
- [ 2 - 电影评分数据集](#2)
  - [ 2.1 用神经网络做基于内容的过滤](#2.1)
  - [ 2.2 准备训练数据](#2.2)
- [ 3 - 用于基于内容过滤的神经网络](#3)
  - [ 3.1 预测](#3.1)
    - [ 练习 1](#ex01)
- [ 4 - 恭喜！](#4)


<a name="1"></a>
## 1 - 程序包 <img align="left" src="./图片/movie_camera.png"     style=" width:40px;  ">
我们将使用熟悉的程序包 NumPy、TensorFlow，以及 [scikit-learn](https://scikit-learn.org/stable/) 中的实用例程。我们还会用 [tabulate](https://pypi.org/project/tabulate/) 来整齐地打印表格，用 [Pandas](https://pandas.pydata.org/) 来组织表格数据。

In [ ]:
import numpy as np
import numpy.ma as ma
from numpy import genfromtxt
from collections import defaultdict
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import tabulate
from recsysNN_utils import *
pd.set_option("display.precision", 1)

<a name="2"></a>
## 2 - 电影评分数据集 <img align="left" src="./图片/film_rating.png" style=" width:40px;" >
该数据集源自 [MovieLens ml-latest-small](https://grouplens.org/datasets/movielens/latest/) 数据集。

[F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

原始数据集包含 600 位用户对 9000 部电影的评分，评分范围为 0.5 到 5，步长 0.5。为聚焦 2000 年以后的电影和热门类型，数据集已做过缩减。缩减后的数据集有 $n_u = 395$ 位用户、$n_m= 694$ 部电影。对每部电影，数据集提供片名、上映日期和一个或多个类型。例如“Toy Story 3”上映于 2010 年，类型为“Adventure|Animation|Children|Comedy|Fantasy|IMAX”。该数据集除了评分之外，关于用户的信息很少。该数据集用于为下文描述的神经网络创建训练向量。

<a name="2.1"></a>
### 2.1 用神经网络做基于内容的过滤

在协同过滤实验中，你生成了两组向量：一个用户向量和一个物品/电影向量，它们的点积可预测评分。这些向量完全由评分推导而来。

基于内容的过滤同样生成用户和电影的特征向量，但它认识到，关于用户和/或电影可能还有其他可用信息可以改进预测。这些额外信息被送入神经网络，由网络生成用户向量和电影向量，如下所示。
<figure>
    <center> <img src="./图片/RecSysNN.png"   style="width:500px;height:280px;" >

> **图中文字（中文）**：神经网络结构示意；右侧标注为“预测评分（Predicted Rating）”。</center>
</figure>
提供给网络的电影内容，是原始数据与一些“构造特征（engineered features）”的组合。回忆课程 1 第 2 周实验 4 中关于特征工程的讨论与实验。原始特征是电影上映年份，以及以独热向量表示的电影类型。共有 14 种类型。构造特征是依据用户评分算出的平均评分。属于多个类型的电影，对每个类型都有一个训练向量。

用户内容只由构造特征组成。系统会为每位用户计算按类型划分的平均评分。此外，还有用户 id、评分次数和平均评分，但它们不包含在训练或预测的内容中，仅用于解读数据。

训练集由数据集中所有用户给出的全部评分构成。用户向量和电影/物品向量一起作为训练集送入上面的网络。对某位用户评分过的所有电影，其用户向量都相同。

下面，让我们加载并显示部分数据。

In [ ]:
# 加载数据，设置配置变量
item_train, user_train, y_train, item_features, user_features, item_vecs, movie_dict, user_to_genre = load_data()

num_user_features = user_train.shape[1] - 3  # 训练时去掉 userid、评分数量和平均评分
num_item_features = item_train.shape[1] - 1  # 训练时去掉电影 id
uvs = 3  # 用户类型向量开始
ivs = 3  # 物品类型向量开始
u_s = 3  # 训练中使用的列区间起点，用户
i_s = 1  # 训练中使用的列区间起点，物品
scaledata = True  # 若为真，则对数据应用标准缩放
print(f"Number of training vectors: {len(item_train)}")

有些用户和物品/电影特征在训练中并不使用。下面，方括号“[]”中的特征，如“user id”“rating count”和“rating ave”，在训练和使用模型时都不包含在内。注意，用户向量对该用户评分过的所有电影都相同。

In [ ]:
pprint_train(user_train, user_features, uvs,  u_s, maxcount=5)

In [ ]:
pprint_train(item_train, item_features, ivs, i_s, maxcount=5, user=False)

In [ ]:
print(f"y_train[:5]: {y_train[:5]}")

从上文可以看到，电影 6874 是一部 2003 年上映的动作片。用户 2 给动作片的平均评分是 3.9。此外，电影 6874 也被列入犯罪（Crime）和惊悚（Thriller）类型。MovieLens 用户给这部电影的平均评分是 4。一个训练样本由两张表中的一行以及 y_train 中的一个评分构成。

<a name="2.2"></a>
### 2.2 准备训练数据
回忆课程 1 第 2 周中你探讨过的特征缩放，它是改善收敛的一种手段。我们将用 [scikit learn StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) 对输入特征做缩放。这在课程 1 第 2 周实验 5 中用过。下面也展示了 inverse_transform，用于还原出原始输入。

In [ ]:
# 缩放训练数据
if scaledata:
    item_train_save = item_train
    user_train_save = user_train

    scalerItem = StandardScaler()
    scalerItem.fit(item_train)
    item_train = scalerItem.transform(item_train)

    scalerUser = StandardScaler()
    scalerUser.fit(user_train)
    user_train = scalerUser.transform(user_train)

    print(np.allclose(item_train_save, scalerItem.inverse_transform(item_train)))
    print(np.allclose(user_train_save, scalerUser.inverse_transform(user_train)))

为便于评估结果，我们将按课程 2 第 3 周讲过的方式把数据划分为训练集和测试集。这里我们使用 [sklean train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) 来划分并打乱数据。注意，把初始随机状态设为同一个值，可确保物品、用户和 y 以完全相同的方式打乱。

In [ ]:
item_train, item_test = train_test_split(item_train, train_size=0.80, shuffle=True, random_state=1)
user_train, user_test = train_test_split(user_train, train_size=0.80, shuffle=True, random_state=1)
y_train, y_test       = train_test_split(y_train,    train_size=0.80, shuffle=True, random_state=1)
print(f"movie/item training data shape: {item_train.shape}")
print(f"movie/item test  data shape: {item_test.shape}")

缩放并打乱后的数据现在均值为零。

In [ ]:
pprint_train(user_train, user_features, uvs, u_s, maxcount=5)

用 Min Max Scaler 对目标评分做缩放，把目标缩放到 -1 到 1 之间。我们使用 scikit-learn，因为它有 inverse_transform。[scikit learn MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)

In [ ]:
scaler = MinMaxScaler((-1, 1))
scaler.fit(y_train.reshape(-1, 1))
ynorm_train = scaler.transform(y_train.reshape(-1, 1))
ynorm_test = scaler.transform(y_test.reshape(-1, 1))
print(ynorm_train.shape, ynorm_test.shape)

<a name="3"></a>
## 3 - 用于基于内容过滤的神经网络
现在，让我们按上图所述构建一个神经网络。它包含两个网络，通过点积合并。你将构建这两个网络。在这个例子中，它们完全相同。注意，这两个网络不必相同。如果用户内容比电影内容大得多，你可能会选择让用户网络的复杂度高于电影网络。在本例中，两边内容相似，因此两个网络相同。

- 使用 Keras 顺序模型
    - 第一层是含 256 个单元的稠密层，激活函数为 relu。
    - 第二层是含 128 个单元的稠密层，激活函数为 relu。
    - 第三层是含 `num_outputs` 个单元的稠密层，激活函数为线性或不用激活函数。
    
网络的其余部分将直接给出。给出的代码没有使用 Keras 顺序模型，而是使用 Keras 的[函数式 API](https://keras.io/guides/functional_api/)。这种形式在组件如何互联上更灵活。


In [ ]:
# 计分单元
# UNQ_C1

num_outputs = 32
tf.random.set_seed(1)
user_NN = tf.keras.models.Sequential([
    ### 在此处编写代码 ###
      
    ### 代码编写结束 ###
])

item_NN = tf.keras.models.Sequential([
    ### 在此处编写代码 ###

    ### 代码编写结束 ###
])

# 创建用户输入并指向基础网络
input_user = tf.keras.layers.Input(shape=(num_user_features))
vu = user_NN(input_user)
vu = tf.linalg.l2_normalize(vu, axis=1)

# 创建物品输入并指向基础网络
input_item = tf.keras.layers.Input(shape=(num_item_features))
vm = item_NN(input_item)
vm = tf.linalg.l2_normalize(vm, axis=1)

# 计算向量 vu 与 vm 的点积
output = tf.keras.layers.Dot(axes=1)([vu, vm])

# 指定模型的输入和输出
model = Model([input_user, input_item], output)

model.summary()

In [ ]:
# 公开测试
from public_tests import *
test_tower(user_NN)
test_tower(item_NN)

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
  你可以按下例创建一个激活函数为 relu 的稠密层。
    

```python     
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### END CODE HERE ###  
])
```
    
<details>
    <summary><font size="2" color="darkblue"><b> 点击查看答案</b></font></summary>
    

```python 
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])
```

</details>
</details>

    


我们将使用均方误差损失和 Adam 优化器。

In [ ]:
tf.random.set_seed(1)
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt,
              loss=cost_fn)

In [ ]:
tf.random.set_seed(1)
model.fit([user_train[:, u_s:], item_train[:, i_s:]], ynorm_train, epochs=30)

在测试数据上评估模型以确定损失。它与训练损失相当，说明模型没有明显过拟合训练数据。

In [ ]:
model.evaluate([user_test[:, u_s:], item_test[:, i_s:]], ynorm_test)

<a name="3.1"></a>
### 3.1 预测
下面，你将用你的模型在若干情形下做预测。
#### 对新用户的预测
首先，我们将创建一个新用户，让模型为该用户推荐电影。在你用示例用户内容试过之后，可以随意把用户内容改成符合你自己偏好的内容，看看模型会推荐什么。注意，评分在 0.5 到 5.0 之间（含端点），步长为 0.5。

In [ ]:
new_user_id = 5000
new_rating_ave = 1.0
new_action = 1.0
new_adventure = 1
new_animation = 1
new_childrens = 1
new_comedy = 5
new_crime = 1
new_documentary = 1
new_drama = 1
new_fantasy = 1
new_horror = 1
new_mystery = 1
new_romance = 5
new_scifi = 5
new_thriller = 1
new_rating_count = 3

user_vec = np.array([[new_user_id, new_rating_count, new_rating_ave,
                      new_action, new_adventure, new_animation, new_childrens,
                      new_comedy, new_crime, new_documentary,
                      new_drama, new_fantasy, new_horror, new_mystery,
                      new_romance, new_scifi, new_thriller]])


让我们看看给这个新用户评分最高的电影。回忆一下，该用户向量偏好的类型是喜剧（Comedy）和爱情（Romance）。
下面，我们将使用一组电影/物品向量 `item_vecs`，它为训练/测试集中的每部电影各有一个向量。把它与上面的用户向量匹配，再用缩放后的向量预测上述新用户对所有电影的评分。

In [ ]:
# 生成用户向量并复制，使其与数据集中的电影数一致。
user_vecs = gen_user_vecs(user_vec,len(item_vecs))

# 缩放向量并为所有电影做预测。返回按评分排序的结果。
sorted_index, sorted_ypu, sorted_items, sorted_user = predict_uservec(user_vecs,  item_vecs, model, u_s, i_s, 
                                                                       scaler, scalerUser, scalerItem, scaledata=scaledata)

print_pred_movies(sorted_ypu, sorted_user, sorted_items, movie_dict, maxcount = 10)

如果你确实在上面创建了一个用户，值得注意的是，网络是以“给定包含用户各类型评分的**一组**评分的用户向量来预测用户评分”为目标训练的。如果数据集中没有与之相似的评分组合的用户，那么简单地给单个类型最高评分、其余类型最低评分，对网络来说可能没有意义。

#### 对已有用户的预测
让我们看看对“用户 36”的预测，他是数据集中的用户之一。我们可以把预测评分与模型的评分做比较。注意，属于多个类型的电影会在训练数据中出现多次。例如，'The Time Machine' 有三个类型：Adventure、Action、Sci-Fi

In [ ]:
uid =  36 
# 构造一组用户向量。这是同一个向量经过变换并重复而成。
user_vecs, y_vecs = get_user_vecs(uid, scalerUser.inverse_transform(user_train), item_vecs, user_to_genre)

# 缩放向量并为所有电影做预测。返回按评分排序的结果。
sorted_index, sorted_ypu, sorted_items, sorted_user = predict_uservec(user_vecs, item_vecs, model, u_s, i_s, scaler, 
                                                                      scalerUser, scalerItem, scaledata=scaledata)
sorted_y = y_vecs[sorted_index]

#打印排序后的预测
print_existing_user(sorted_ypu, sorted_y.reshape(-1,1), sorted_user, sorted_items, item_features, ivs, uvs, movie_dict, maxcount = 10)

#### 寻找相似物品
上面的神经网络产生两组特征向量：用户特征向量 $v_u$ 和电影特征向量 $v_m$。它们是 32 维向量，其数值难以解释。不过，相似的物品会有相似的向量。这一信息可用于生成推荐。例如，如果某用户给“Toy Story 3”打了高分，就可以通过选取电影特征向量相似的电影来推荐同类影片。

一种相似度度量是两个向量 $ \mathbf{v_m^{(k)}}$ 和 $\mathbf{v_m^{(i)}}$ 之间的平方距离：
$$\left\Vert \mathbf{v_m^{(k)}} - \mathbf{v_m^{(i)}}  \right\Vert^2 = \sum_{l=1}^{n}(v_{m_l}^{(k)} - v_{m_l}^{(i)})^2\tag{1}$$

<a name="ex01"></a>
### 练习 1

编写一个函数来计算平方距离。

In [ ]:
# 计分函数：sq_dist
# UNQ_C2
def sq_dist(a,b):
    """返回两个向量之间的平方距离
    参数：
      a (ndarray (n,)): 含 n 个特征的向量
      b (ndarray (n,)): 含 n 个特征的向量
    返回：
      d (float) : 距离"""
    ### 在此处编写代码 ###
    
    ### 代码编写结束 ###
    return (d)

In [ ]:
# 公开测试
test_sq_dist(sq_dist)

In [ ]:
a1 = np.array([1.0, 2.0, 3.0]); b1 = np.array([1.0, 2.0, 3.0])
a2 = np.array([1.1, 2.1, 3.1]); b2 = np.array([1.0, 2.0, 3.0])
a3 = np.array([0, 1, 0]);       b3 = np.array([1, 0, 0])
print(f"squared distance between a1 and b1: {sq_dist(a1, b1)}")
print(f"squared distance between a2 and b2: {sq_dist(a2, b2)}")
print(f"squared distance between a3 and b3: {sq_dist(a3, b3)}")

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
  求和通常意味着应该用 for 循环，但这里的减法可以用一条语句按元素完成。此外，你可以用 np.square 对相减的结果按元素求平方，用 np.sum 对平方后的元素求和。
    
</details>

    


电影之间的距离矩阵可以在模型训练完成后计算一次，之后用于新的推荐而无需重新训练。模型训练完成后，第一步是取得每部电影的电影特征向量 $v_m$。为此，我们将使用训练好的 `item_NN`，并构建一个小模型，让我们能把电影向量送进去以生成 $v_m$。

In [ ]:
input_item_m = tf.keras.layers.Input(shape=(num_item_features))    # 输入层
vm_m = item_NN(input_item_m)                                       # 使用训练好的 item_NN
vm_m = tf.linalg.l2_normalize(vm_m, axis=1)                        # 像原始模型中那样加入归一化
model_m = Model(input_item_m, vm_m)                                
model_m.summary()

有了电影模型之后，你就可以用一组物品/电影向量作为输入，让模型做预测，从而得到一组电影特征向量。`item_vecs` 是所有电影向量的集合。回忆一下，同一部电影会按它的每个类型分别出现为一个向量。要配合训练好的模型使用，必须对它做缩放。预测结果就是每部电影的一个 32 维特征向量。

In [ ]:
scaled_item_vecs = scalerItem.transform(item_vecs)
vms = model_m.predict(scaled_item_vecs[:,i_s:])
print(f"size of all predicted movie feature vectors: {vms.shape}")

现在让我们计算每部电影特征向量与所有其他电影特征向量之间的平方距离矩阵：
<figure>
    <left> <img src="./图片/distmatrix.PNG"   style="width:400px;height:225px;" >

> **图中文字（中文）**：d(0,1) d(0,2) …… d(0,m-1) …… 求最小值（Find min）</center>
</figure>

然后，我们可以沿每一行取最小值，找出最接近的电影。我们将利用 [numpy 掩码数组](https://numpy.org/doc/1.21/user/tutorial-ma.html)来避免选到同一部电影。对角线上的被掩码的值不会参与计算。

In [ ]:
count = 50
dim = len(vms)
dist = np.zeros((dim,dim))

for i in range(dim):
    for j in range(dim):
        dist[i,j] = sq_dist(vms[i, :], vms[j, :])
        
m_dist = ma.masked_array(dist, mask=np.identity(dist.shape[0]))  # 屏蔽对角线

disp = [["movie1", "genres", "movie2", "genres"]]
for i in range(count):
    min_idx = np.argmin(m_dist[i])
    movie1_id = int(item_vecs[i,0])
    movie2_id = int(item_vecs[min_idx,0])
    genre1,_  = get_item_genre(item_vecs[i,:], ivs, item_features)
    genre2,_  = get_item_genre(item_vecs[min_idx,:], ivs, item_features)

    disp.append( [movie_dict[movie1_id]['title'], genre1,
                  movie_dict[movie2_id]['title'], genre2]
               )
table = tabulate.tabulate(disp, tablefmt='html', headers="firstrow", floatfmt=[".1f", ".1f", ".0f", ".2f", ".2f"])
table

结果表明，模型会推荐同一类型的电影。

<a name="4"></a>
## 4 - 恭喜！ <img align="left" src="./图片/film_award.png" style=" width:40px;">
你完成了一个基于内容的推荐系统。

这种结构是许多商业推荐系统的基础。只要数据可得，用户内容可以大幅扩充，纳入更多关于用户的信息。物品不限于电影。它可用于推荐任何物品：书籍、汽车，或与你“购物车”中某件物品相似的东西。